# Cats vs Dogs Classification Using CNN

This notebook prepares the Oxford-IIIT Pet images, creates binary labels, splits the dataset, and trains a Convolutional Neural Network (CNN). Only the parts needed for correct training and evaluation have been adjusted.


## 1. Imports


In [ ]:
import os
import tarfile
import cv2
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

## 2. Check the Dataset Folder
If the images have already been extracted, this cell checks that the expected folder exists.


In [ ]:
import os

extract_path = "data/images/images"

if os.path.exists(extract_path):
    print("Dataset folder exists!")
else:
    print("Dataset folder not found!")

## 3. Collect Image Paths
Collect image files only and sort them so that image order remains consistent. Files such as `.mat` are not image files and are excluded.


In [ ]:
image_paths = []

for root, dirs, files in os.walk(extract_path):
    for file in files:
        if file.lower().endswith((".jpg", ".jpeg", ".png")):
            image_paths.append(os.path.join(root, file))

image_paths = sorted(image_paths)

print("Total images:", len(image_paths))
print("First 5 files:")

for path in image_paths[:5]:
    print(os.path.basename(path))

## 4. Display Sample Images
Display up to 20 images to visually inspect the dataset.


In [ ]:
plt.figure(figsize=(15, 8))

for i in range(min(20, len(image_paths))):
    img = Image.open(image_paths[i])

    plt.subplot(4, 5, i + 1)
    plt.imshow(img)
    plt.title(os.path.basename(image_paths[i]))
    plt.axis("off")

plt.tight_layout()
plt.show()

## 5. Resize and Reshape Images
Convert each image to grayscale and resize it to `130 × 130` pixels. Keep track of successfully loaded images so their labels match the images in `X`.


In [ ]:
W, H = 130, 130
Allimages = []
valid_paths = []

for path in image_paths:
    img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)

    if img is not None:
        img = cv2.resize(img, (W, H))
        img = img.reshape(W, H, 1)
        Allimages.append(img)
        valid_paths.append(path)
    else:
        print("Could not read:", path)

X = np.array(Allimages, dtype=np.uint8)

print("X shape:", X.shape)
print("X data type:", X.dtype)


## 6. Create Binary Labels (`y`)
For Oxford-IIIT Pet filenames, cat breed names start with an uppercase letter and dog breed names start with a lowercase letter. Labels are `0 = Cat` and `1 = Dog`. Labels are generated from the same list of successfully loaded images used to create `X`.


In [ ]:
y = []

for path in valid_paths:
    filename = os.path.basename(path)
    breed = filename.split("_")[0]

    if breed[0].isupper():
        y.append(0)  # Cat
    else:
        y.append(1)  # Dog

y = np.array(y, dtype=np.int32)

print("X samples:", len(X))
print("y labels:", len(y))
print("Cat labels (0):", np.sum(y == 0))
print("Dog labels (1):", np.sum(y == 1))


## 7. Check That Images and Labels Match
The number of images must equal the number of labels before training.


In [ ]:
assert len(X) == len(y), "X and y have different lengths!"

print("X and y match.")

## 8. Train / Validation / Test Split
Split the dataset into 70% training, 15% validation, and 15% test data. `stratify` keeps the cat/dog proportions approximately consistent across the splits.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

X_val, X_test, y_val, y_test = train_test_split(
    X_test, y_test,
    test_size=0.50,
    random_state=42,
    stratify=y_test
)

print("Training:", X_train.shape, y_train.shape)
print("Validation:", X_val.shape, y_val.shape)
print("Testing:", X_test.shape, y_test.shape)

## 9. Important Notes
- `X` contains the images and `y` contains their binary labels.
- The output layer uses `sigmoid` because this is a binary classification problem with one output neuron.
- Pixel values are scaled to the range `[0, 1]` inside the model.
- A higher accuracy cannot be guaranteed; review the training and validation curves to see whether the model is learning or overfitting.


In [ ]:
model = keras.Sequential([
    keras.layers.Input(shape=(W, H, 1)),
    keras.layers.Rescaling(1.0 / 255),
    keras.layers.Conv2D(32, 3, padding="valid", activation="relu"),
    keras.layers.MaxPooling2D(pool_size=(2, 2)),
    keras.layers.Conv2D(64, 3, padding="valid", activation="relu"),
    keras.layers.MaxPooling2D(pool_size=(2, 2)),
    keras.layers.Conv2D(128, 3, padding="valid", activation="relu"),
    keras.layers.MaxPooling2D(pool_size=(2, 2)),
    keras.layers.Flatten(),
    keras.layers.Dense(128, activation="relu"),
    keras.layers.Dropout(0.3),
    keras.layers.Dense(1, activation="sigmoid")
])


In [ ]:
model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)


In [ ]:
history = model.fit(
    X_train,
    y_train,
    epochs=50,
    batch_size=32,
    validation_data=(X_val, y_val),
    verbose=1
)

val_loss, val_accuracy = model.evaluate(X_val, y_val, verbose=0)
print("Validation Accuracy:", val_accuracy)
print("Validation Loss:", val_loss)


## 10. Test Evaluation and Predictions
Evaluate the trained model on the separate test set. The predicted probability is converted to a class using a threshold of `0.5`: `0 = Cat`, `1 = Dog`.

In [ ]:
test_loss, test_accuracy = model.evaluate(X_test, y_test, verbose=0)
print("Test Accuracy:", test_accuracy)
print("Test Loss:", test_loss)

predicted_probabilities = model.predict(X_test[:10], verbose=0).flatten()
predicted_labels = (predicted_probabilities >= 0.5).astype(np.int32)

for i, (probability, predicted_label, true_label) in enumerate(zip(predicted_probabilities, predicted_labels, y_test[:10])):
    predicted_class = "Dog" if predicted_label == 1 else "Cat"
    true_class = "Dog" if true_label == 1 else "Cat"
    print(f"Image {i + 1}: Predicted = {predicted_class} ({probability:.3f}), Actual = {true_class}")
